In [ ]:
# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Create dataset directory and copy files from Drive
!mkdir -p "/content/dataset"
!cp -r "/content/drive/MyDrive/ppe detection.yolov8/"* "/content/dataset/"

In [ ]:
# Verify the dataset structure
!ls "/content/drive/MyDrive/ppe detection.yolov8"

In [ ]:
import shutil
import os

# Source and destination paths
source_dir = "/content/drive/MyDrive/ppe detection.yolov8"
target_dir = "/content/dataset"

# Safe copy with overwrite
if os.path.exists(source_dir):
    shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)
    print("Dataset copied successfully!")
    print("Target directory contents:", os.listdir(target_dir))

    train_path = os.path.join(target_dir, "train")
    if os.path.exists(train_path):
        files = os.listdir(train_path)
        print("Train folder contents:", files[:5], "... (total files:", len(files), ")")
else:
    print("Source directory not found. Please check the Drive path.")

In [ ]:
import os
import random
import shutil

base_dir = "/content/dataset"
train_img_dir = os.path.join(base_dir, "train/images")
train_lbl_dir = os.path.join(base_dir, "train/labels")

# Create validation and test directories
for split in ["valid", "test"]:
    os.makedirs(os.path.join(base_dir, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(base_dir, split, "labels"), exist_ok=True)

# List all images and shuffle with fixed seed for reproducibility
all_images = sorted([f for f in os.listdir(train_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
random.seed(42)
random.shuffle(all_images)

total_files = len(all_images)
num_val  = int(total_files * 0.10)
num_test = int(total_files * 0.10)

val_images  = all_images[:num_val]
test_images = all_images[num_val:num_val + num_test]

print("Splitting dataset...")

# Move validation images
for img in val_images:
    lbl = os.path.splitext(img)[0] + ".txt"
    shutil.move(os.path.join(train_img_dir, img), os.path.join(base_dir, "valid/images", img))
    if os.path.exists(os.path.join(train_lbl_dir, lbl)):
        shutil.move(os.path.join(train_lbl_dir, lbl), os.path.join(base_dir, "valid/labels", lbl))

# Move test images
for img in test_images:
    lbl = os.path.splitext(img)[0] + ".txt"
    shutil.move(os.path.join(train_img_dir, img), os.path.join(base_dir, "test/images", img))
    if os.path.exists(os.path.join(train_lbl_dir, lbl)):
        shutil.move(os.path.join(train_lbl_dir, lbl), os.path.join(base_dir, "test/labels", lbl))

print("\nDataset split complete!")
print("Train images :", len(os.listdir(os.path.join(base_dir, "train/images"))))
print("Valid images :", len(os.listdir(os.path.join(base_dir, "valid/images"))))
print("Test  images :", len(os.listdir(os.path.join(base_dir, "test/images"))))

In [ ]:
import os
import torch
import cv2
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


class PPEYoloDataset(Dataset):
    """
    Custom Dataset that reads YOLO-format annotations and converts them
    on-the-fly to the COCO format expected by Faster R-CNN.

    Label mapping:
        YOLO class 0 (Helmet)    -> Faster R-CNN class 1
        YOLO class 1 (No_Helmet) -> Faster R-CNN class 2
        Class 0 is always reserved for background in Faster R-CNN.
    """

    def __init__(self, image_dir, label_dir, transforms=None):
        self.image_dir   = image_dir
        self.label_dir   = label_dir
        self.transforms  = transforms
        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.endswith(('.jpg', '.jpeg', '.png'))
        ])

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        height, width, _ = img.shape

        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_name)

        boxes  = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.split()
                    if len(parts) == 5:
                        # Shift class IDs by 1 (background = 0 in Faster R-CNN)
                        class_id = int(parts[0]) + 1

                        # Convert YOLO normalised [cx, cy, w, h] -> [xmin, ymin, xmax, ymax]
                        x_c, y_c, w, h = map(float, parts[1:])
                        xmin = (x_c - w / 2) * width
                        ymin = (y_c - h / 2) * height
                        xmax = (x_c + w / 2) * width
                        ymax = (y_c + h / 2) * height

                        if xmax > xmin and ymax > ymin:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(class_id)

        # Avoid errors on empty images by adding a dummy background box
        if len(boxes) == 0:
            boxes  = [[0, 0, 1, 1]]
            labels = [0]

        boxes     = torch.as_tensor(boxes,  dtype=torch.float32)
        labels    = torch.as_tensor(labels, dtype=torch.int64)
        image_id  = torch.tensor([idx])
        area      = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd   = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes"   : boxes,
            "labels"  : labels,
            "image_id": image_id,
            "area"    : area,
            "iscrowd" : iscrowd
        }

        img_tensor = torchvision.transforms.functional.to_tensor(img)
        return img_tensor, target

    def __len__(self):
        return len(self.image_files)


def collate_fn(batch):
    return tuple(zip(*batch))


print("Dataset class defined successfully!")

In [ ]:
# Create DataLoaders for train and validation sets
# Batch size 4 is safe for T4 GPU memory
train_dataset = PPEYoloDataset("/content/dataset/train/images", "/content/dataset/train/labels")
valid_dataset = PPEYoloDataset("/content/dataset/valid/images", "/content/dataset/valid/labels")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print("Train and Validation DataLoaders are ready!")

In [ ]:
# Load pre-trained Faster R-CNN with ResNet-50 FPN backbone
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.COCO_V1
)

# Number of classes: 2 (Helmet, No_Helmet) + 1 (background) = 3
num_classes = 3

# Replace the classifier head with a new one for our class count
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Move model to GPU if available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

print(f"Model loaded and moved to {device}!")

In [ ]:
# Optimiser: SGD with momentum and weight decay (standard for Faster R-CNN fine-tuning)
params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 10
print("Training started — epoch loss will be printed after each epoch.")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for images, targets in train_loader:
        images  = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # In training mode Faster R-CNN returns a dict of losses
        loss_dict = model(images, targets)
        losses    = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{num_epochs}]  -  Average Loss: {avg_loss:.4f}")

print("Training complete!")

# Save model weights
torch.save(model.state_dict(), "faster_rcnn_ppe.pth")
print("Model weights saved as 'faster_rcnn_ppe.pth'")

In [ ]:
!pip install -q torchmetrics

In [ ]:
import time
import torch
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Prepare test DataLoader
test_dataset = PPEYoloDataset("/content/dataset/test/images", "/content/dataset/test/labels")
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print("Evaluating on test set (mAP and FPS)...")
model.eval()

# Initialise mAP metric
metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')

total_time   = 0
total_images = 0

with torch.no_grad():
    for images, targets in test_loader:
        images = list(image.to(device) for image in images)

        start_time = time.time()
        outputs    = model(images)

        # Synchronise CUDA before stopping the timer for accurate FPS measurement
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end_time = time.time()

        total_time   += (end_time - start_time)
        total_images += len(images)

        preds = [
            {
                "boxes" : out["boxes"].cpu(),
                "scores": out["scores"].cpu(),
                "labels": out["labels"].cpu()
            }
            for out in outputs
        ]

        target_list = [
            {
                "boxes" : t["boxes"].cpu(),
                "labels": t["labels"].cpu()
            }
            for t in targets
        ]

        metric.update(preds, target_list)

# Compute final results
results = metric.compute()
fps     = total_images / total_time

print("\n" + "=" * 50)
print("FASTER R-CNN TEST RESULTS")
print("=" * 50)
print(f"Test images evaluated : {total_images}")
print(f"mAP@50 (YOLO equiv.)  : {results['map_50'].item() * 100:.2f}%")
print(f"Inference speed (FPS) : {fps:.2f}  (YOLOv8s = 103 FPS)")
print("=" * 50)